# Two Pointers and Sliding Window
## Week 2 · Beginner-friendly team lesson

**Your goal:** Recognise when a sorted or contiguous problem can be solved with a one-pass scan, explain why the scan works, and test a solution.

By the end, you should be able to:
1. Use two pointers to find a target pair in a **sorted** list.
2. Use a **fixed-size window** to analyse consecutive daily loan counts.
3. Use a **variable-size window** to find a shortest interval with a target total.
4. Explain when these methods do **not** apply.

**You need:** Python lists, indexes, `if`, `for`, `while`, and functions. All code uses built-in Python. Run cells from top to bottom with **Shift + Enter**.

**Presenter:** Allow about 75–90 minutes. Ask for predictions before running code. Learners can trace every example on paper before coding.

## 1. The problem: avoid doing the same work again

Suppose daily loans are `[2, 1, 5, 1, 3, 2]`. We want the busiest **three consecutive days**.

A direct solution totals each three-day block from scratch:
- Days 1–3: `2 + 1 + 5 = 8`
- Days 2–4: `1 + 5 + 1 = 7`

The blocks overlap. Instead of adding the shared values again, remove the old value and add the new one: **`8 - 2 + 1 = 7`**.

Two-pointer methods also reuse progress: after checking a pair, they move an index in a direction that safely rules out more candidates.

**Question for the class:** What information can we keep instead of restarting the search?

In [3]:
# Prerequisite refresher: an index is a zero-based position.
loans = [2, 1, 5, 1, 3, 2]
print("First value:", loans[0])
print("Last value:", loans[len(loans) - 1])
print("Positions 0, 1, 2:", loans[0:3])  # Stop index is excluded.
print("Total for those positions:", sum(loans[0:3]))

left = 1
right = 3
print("Inclusive interval 1 through 3:", loans[left:right + 1])
print("Length of inclusive interval:", right - left + 1)

First value: 2
Last value: 2
Positions 0, 1, 2: [2, 1, 5]
Total for those positions: 8
Inclusive interval 1 through 3: [1, 5, 1]
Length of inclusive interval: 3


## 2. Three terms we must agree on

| Term | Meaning | Example |
|---|---|---|
| **Sorted** | Values are in non-decreasing order. | `[1, 2, 2, 7]` |
| **Contiguous** | Values occupy consecutive positions; nothing between them is skipped. | In `[2, 1, 5, 1]`, `[1, 5]` is contiguous. Picking positions 0 and 2 skips position 1. |
| **Pointer** | An integer index marking a position. | `left = 0`, `right = 4` |

### What does “one pass” mean here?
Each pointer only moves forward in its chosen direction. It does not restart for every candidate. Two pointers can make up to roughly `2n` moves and still take **O(n)** time.

**O(n)** means work grows roughly in proportion to the number of items. **O(n²)** means doubling the input can produce roughly four times the work.

A nested loop is not automatically O(n²): count the **total pointer movements**.

**Check:** Is every pair of items a contiguous interval? **No.** A target-pair problem and a window problem ask different questions.

## 3. Two pointers: find a pair in a sorted list
### Restate and examples

Write `find_pair_sorted(numbers, target)`.
- Input is a list of numbers sorted in non-decreasing order.
- Return a pair of **different indexes** whose values sum to the target.
- Return `None` if no pair exists. Any valid pair is acceptable.
- Do not change the input. Duplicate values are allowed, but one position cannot be used twice.

| Numbers | Target | One correct result |
|---|---:|---|
| `[1, 2, 4, 6, 10]` | 8 | `(1, 3)` because `2 + 6 = 8` |
| `[2, 2]` | 4 | `(0, 1)` |
| `[1, 3, 5]` | 10 | `None` — there is only one 5 |
| `[]` | 8 | `None` |

The pair need not be adjacent. **Your turn:** Find the answer to the first row by hand.

## 4. Why moving a pointer is safe

Start at the smallest and largest values:
```text
indexes:  0   1   2   3    4
values:   1   2   4   6   10
          L               R
```
- **Sum too small:** Move `left` right to try a larger value.
- **Sum too large:** Move `right` left to try a smaller value.
- **Sum equal:** Return the indexes.

### Why this does not skip an answer
If `numbers[left] + numbers[right]` is too small, pairing that left value with any smaller remaining right value is also too small. We can discard that left position.

If the sum is too large, pairing that right value with any larger remaining left value is also too large. We can discard that right position.

**This reasoning depends on sorted order.** Merely using two indexes does not make a solution correct.

## 5. Trace the plan before coding
Target: **8**. List: `[1, 2, 4, 6, 10]`.

| Left index/value | Right index/value | Sum | Action |
|---|---|---:|---|
| 0 / 1 | 4 / 10 | 11 | Too large → right becomes 3 |
| 0 / 1 | 3 / 6 | 7 | Too small → left becomes 1 |
| 1 / 2 | 3 / 6 | 8 | Return `(1, 3)` |

### Plan
1. Set `left` to 0 and `right` to the last index.
2. While `left < right`, calculate their sum.
3. Return the indexes if it matches the target.
4. Move left for a small sum; move right for a large sum.
5. If the indexes meet, return `None`.

**Why `<`, not `<=`?** We need two different positions.

**Reflect:** We discarded an endpoint only when sorted order proved it could not be part of a solution in the remaining range.

In [4]:
def find_pair_sorted(numbers, target):
    """Return two distinct indexes, or None. Input must already be sorted."""
    left = 0
    right = len(numbers) - 1

    while left < right:
        pair_sum = numbers[left] + numbers[right]
        if pair_sum == target:
            return (left, right)
        if pair_sum < target:
            left += 1
        else:
            right -= 1

    return None

numbers = [1, 2, 4, 6, 10]
pair = find_pair_sorted(numbers, 8)
print("Indexes:", pair)
if pair is not None:
    print("Values:", numbers[pair[0]], numbers[pair[1]])

Indexes: (1, 3)
Values: 2 6


In [7]:
# Test the rules, not one particular choice when multiple pairs are valid.
pair_cases = [
    ([1, 2, 4, 6, 10], 8, True),
    ([2, 2], 4, True),
    ([1, 3, 5], 10, False),
    ([], 8, False),
    ([4], 8, False),
    ([-5, -2, 0, 3, 8], 1, True),
]
for numbers, target, exists in pair_cases:
    result = find_pair_sorted(numbers, target)
    assert (result is not None) == exists
    if result is not None:
        left, right = result
        assert 0 <= left < right < len(numbers)
        assert numbers[left] + numbers[right] == target
    print(f"PASS: {numbers}, target={target}, result={result}")

PASS: [1, 2, 4, 6, 10], target=8, result=(1, 3)
PASS: [2, 2], target=4, result=(0, 1)
PASS: [1, 3, 5], target=10, result=None
PASS: [], target=8, result=None
PASS: [4], target=8, result=None
PASS: [-5, -2, 0, 3, 8], target=1, result=(1, 3)


In [8]:
# Make the pointer movements visible. Predict each move before running.
trace_numbers = [1, 2, 4, 6, 10]
trace_target = 8
left, right = 0, len(trace_numbers) - 1
checks = 0
while left < right:
    total = trace_numbers[left] + trace_numbers[right]
    checks += 1
    print(f"left={left}, right={right}, sum={total}")
    if total == trace_target:
        print("Found a valid pair.")
        break
    if total < trace_target:
        print("Move left rightward.")
        left += 1
    else:
        print("Move right leftward.")
        right -= 1
print("Pairs checked:", checks)


left=0, right=4, sum=11
Move right leftward.
left=0, right=3, sum=7
Move left rightward.
left=1, right=3, sum=8
Found a valid pair.
Pairs checked: 3


## 6. When this pair method does not apply

The pointer scan is **O(n) time** and **O(1) extra space** when the input is already sorted.

If you sort first, the whole method is generally **O(n log n)**, not O(n). Sorting also changes positions; indexes in the sorted list are not the original indexes.

For an unsorted target-pair problem, a dictionary can give an average O(n) solution using O(n) extra space. That is a different method.

**Do not assume every sorted problem has a one-pass solution.** You still need a rule proving which pointer can move safely.

**Your turn:** On `[1, 4, 7, 9]`, target 11, which pointer moves after the first sum? Trace until you find an answer.

In [9]:
# Demonstration of violating the precondition. Do NOT copy this as a valid use.
unsorted_numbers = [3, 2, 4]
result = find_pair_sorted(unsorted_numbers, 6)
print("Incorrectly applied to unsorted input:", result)
print("But positions 1 and 2 contain 2 + 4 = 6.")
assert result is None  # This shows a missed answer, not correctness on unsorted data.

# Sorting allows the scan, but the indexes now refer to sorted_numbers.
sorted_numbers = sorted(unsorted_numbers)
print("Sorted values:", sorted_numbers)
print("Indexes in SORTED list:", find_pair_sorted(sorted_numbers, 6))

Incorrectly applied to unsorted input: None
But positions 1 and 2 contain 2 + 4 = 6.
Sorted values: [2, 3, 4]
Indexes in SORTED list: (0, 2)


## 7. Sliding window: two boundaries around consecutive items

A **window** is a contiguous interval of a list or string. Sliding moves its boundaries without rebuilding all its information.

```text
[2, 1, 5, 1, 3, 2]
 [-----]             first three-day window: 2, 1, 5
    [-----]          next window:            1, 5, 1
```

| Question wording | Likely approach |
|---|---|
| Exactly `k` consecutive items | Fixed-size window |
| Shortest/longest consecutive interval meeting a condition | Consider a variable-size window; first check that moving boundaries has a predictable effect |
| Any two values, not necessarily adjacent, in sorted input | Opposite-end two pointers may fit |

**Contiguous alone is not enough.** The information must be maintainable efficiently as the boundaries move.

## 8. Fixed-size window: busiest consecutive days
### Restate and examples

Write `max_consecutive_total(values, k)`.
- Return the largest sum of **exactly `k` consecutive values**.
- Require an integer `k` with `1 <= k <= len(values)`; invalid size raises `ValueError`.
- Assume numeric values and integer `k`. Values may be negative.

| Values | k | Expected |
|---|---:|---:|
| `[2, 1, 5, 1, 3, 2]` | 3 | 9, from `[5, 1, 3]` |
| `[4, 2, 7]` | 1 | 7 |
| `[4, 2, 7]` | 3 | 13 |
| `[-4, -2, -7]` | 2 | -6 |
| `[]` | 1 | ValueError |

**Library example:** `values` can represent daily loan counts. Here we return the total, not the dates.

In [6]:
# Direct baseline: correct, but it repeats additions for overlapping windows.
def max_consecutive_total_slow(values, k):
    if not 1 <= k <= len(values):
        raise ValueError("k must be between 1 and the list length.")
    best = None
    for start in range(len(values) - k + 1):
        current = sum(values[start:start + k])
        if best is None or current > best:
            best = current
    return best

print("Baseline answer:", max_consecutive_total_slow([2, 1, 5, 1, 3, 2], 3))
# There are n-k+1 windows, each summing k items: O((n-k+1)*k) time.
# The slice also copies k items. We can avoid both repeated sum work and slices.

Baseline answer: 9


## 9. Reuse the previous total

For `[2, 1, 5, 1, 3, 2]` and `k = 3`:

| Window | Remove | Add | New total | Best so far |
|---|---:|---:|---:|---:|
| `[2, 1, 5]` | — | — | 8 | 8 |
| `[1, 5, 1]` | 2 | 1 | 7 | 8 |
| `[5, 1, 3]` | 1 | 3 | 9 | 9 |
| `[1, 3, 2]` | 5 | 2 | 6 | 9 |

### Plan
1. Validate `k`.
2. Add the first `k` values; make this total the current best.
3. For each next index `right`, add `values[right]` and subtract `values[right-k]`.
4. Update the best total if needed.
5. Return the best total.

**Do not start best at zero:** With negative values, every valid window may have a negative total.

**Your turn:** Explain why `right-k` identifies the item that leaves.

In [7]:
def max_consecutive_total(values, k):
    if not 1 <= k <= len(values):
        raise ValueError("k must be between 1 and the list length.")

    # Build the first window without creating a slice.
    window_total = 0
    for index in range(k):
        window_total += values[index]
    best = window_total

    for right in range(k, len(values)):
        window_total += values[right] - values[right - k]
        if window_total > best:
            best = window_total

    return best

print("Busiest three-day total:", max_consecutive_total([2, 1, 5, 1, 3, 2], 3))

Busiest three-day total: 9


In [8]:
fixed_cases = [
    ([2, 1, 5, 1, 3, 2], 3, 9),
    ([4, 2, 7], 1, 7),
    ([4, 2, 7], 3, 13),
    ([-4, -2, -7], 2, -6),
    ([0, 0, 0], 2, 0),
]
for values, k, expected in fixed_cases:
    actual = max_consecutive_total(values, k)
    assert actual == expected
    assert actual == max_consecutive_total_slow(values, k)
    print(f"PASS: values={values}, k={k}, total={actual}")

for values, k in [([], 1), ([1, 2], 0), ([1, 2], 3), ([1, 2], -1)]:
    try:
        max_consecutive_total(values, k)
    except ValueError:
        print(f"PASS: rejected k={k} for {values}")
    else:
        raise AssertionError("Invalid window size was accepted")

PASS: values=[2, 1, 5, 1, 3, 2], k=3, total=9
PASS: values=[4, 2, 7], k=1, total=7
PASS: values=[4, 2, 7], k=3, total=13
PASS: values=[-4, -2, -7], k=2, total=-6
PASS: values=[0, 0, 0], k=2, total=0
PASS: rejected k=1 for []
PASS: rejected k=0 for [1, 2]
PASS: rejected k=3 for [1, 2]
PASS: rejected k=-1 for [1, 2]


## 10. Why the fixed window is one pass

- Initial window: `k` additions.
- Remaining items: `n-k` constant-size updates.
- Total: **O(n) time**, **O(1) extra space**.

The loops are consecutive, not one full scan repeated inside another. Each value enters once and leaves at most once.

**Reflect:** Keeping a running sum removed repeated work. Negative values are fine because the window length is fixed; we never use the sign of a value to decide which boundary to move.

**Important distinction:** Fixed-size sliding sums work with negative numbers. The next variable-size method requires **non-negative** numbers.

**Check:** If `k = 1`, what should the function return? The largest single value.

## 11. Variable-size window: reach a target in the fewest days
### Restate and examples

Write `shortest_target_window(values, target)`.
- Values are **non-negative numbers**. The target is **positive**.
- Return the shortest length of a contiguous interval whose sum is **at least** the target.
- Return `0` if no interval qualifies. Assume these input rules hold.

| Values | Target | Expected |
|---|---:|---|
| `[2, 3, 1, 2, 4, 3]` | 7 | 2, from `[4, 3]` |
| `[8]` | 7 | 1 |
| `[1, 1]` | 7 | 0 |
| `[]` | 7 | 0 |
| `[0, 0, 7]` | 7 | 1 |

### Why non-negative values matter
Adding a value on the right cannot decrease the sum. Removing a value on the left cannot increase it. This gives us a reliable boundary-movement rule.

**Your turn:** Find one qualifying interval in the first row. Is it the shortest?

## 12. Expand to qualify; shrink to improve

### Plan
1. Set `left = 0`, `total = 0`, and no best length yet.
2. Move `right` forward, adding each value to the total.
3. While the total reaches the target:
   - Record the current length if it is shorter.
   - Remove `values[left]` and move `left` forward.
4. Return the best length, or 0 if none qualified.

**Why shrink in a `while`, not just an `if`?** More than one left-hand value may be removable while the remaining window still qualifies.

### Trace: `[2, 3, 1, 2, 4, 3]`, target 7
| Right index | After adding | Qualifying windows tested while shrinking | Best length afterward |
|---:|---:|---|---:|
| 0 | 2 | None | — |
| 1 | 5 | None | — |
| 2 | 6 | None | — |
| 3 | 8 | `[2, 3, 1, 2]`, length 4 | 4 |
| 4 | 10 | `[3, 1, 2, 4]`, length 4; `[1, 2, 4]`, length 3 | 3 |
| 5 | 9 | `[2, 4, 3]`, length 3; `[4, 3]`, length 2 | 2 |

After each row, shrinking stops when the sum falls below 7. The next right-hand item may make it qualify again.

In [ ]:
def shortest_target_window(values, target):
    """Preconditions: non-negative values and a positive target."""
    left = 0
    window_total = 0
    best_length = len(values) + 1  # Bigger than every possible valid length.

    for right in range(len(values)):
        window_total += values[right]

        while window_total >= target:
            current_length = right - left + 1
            if current_length < best_length:
                best_length = current_length
            window_total -= values[left]
            left += 1

    if best_length == len(values) + 1:
        return 0
    return best_length

print("Shortest qualifying length:", shortest_target_window([2, 3, 1, 2, 4, 3], 7))

In [10]:
variable_cases = [
    ([2, 3, 1, 2, 4, 3], 7, 2),
    ([8], 7, 1),
    ([1, 1], 7, 0),
    ([], 7, 0),
    ([0, 0, 7], 7, 1),
    ([2, 2, 2], 4, 2),
]
for values, target, expected in variable_cases:
    actual = shortest_target_window(values, target)
    assert actual == expected
    print(f"PASS: {values}, target={target}, length={actual}")

# A negative value breaks the movement rule. This input is OUTSIDE the contract.
invalid_values = [1, -1, 5]
print("Outside-contract result:", shortest_target_window(invalid_values, 5))
print("Correct shortest length would be 1 from [5], but the scan above returns 3.")
assert shortest_target_window(invalid_values, 5) == 3

PASS: [2, 3, 1, 2, 4, 3], target=7, length=2
PASS: [8], target=7, length=1
PASS: [1, 1], target=7, length=0
PASS: [], target=7, length=0
PASS: [0, 0, 7], target=7, length=1
PASS: [2, 2, 2], target=4, length=2
Outside-contract result: 3
Correct shortest length would be 1 from [5], but the scan above returns 3.


## 13. Why the nested loop is still O(n)

`right` visits each position once. `left` only moves forward and can advance at most `n` times **across the whole function**, not `n` times for every right position.

So total pointer movement is at most about `2n`: **O(n) time**, **O(1) extra space**.

**Why shrinking is safe:** Once a window qualifies, any longer future window with the same left boundary cannot improve its length. We record it, discard that left boundary, and try a shorter candidate.

**Reflect:** The non-negative rule is part of the algorithm, not optional advice. In `[1, -1, 5]`, removing `1` lowers the sum below 5, so the method stops shrinking before discovering `[5]`. Other methods are needed when negatives are allowed.

Also keep `target > 0`: this implementation is not designed for zero or negative targets.

## 14. Choose the method before writing code

| Ask yourself | If yes |
|---|---|
| Do I need two values in an already sorted list, and can a sum comparison safely discard an endpoint? | Opposite-end two pointers |
| Do I need exactly `k` consecutive values and can I update the answer when one enters and one leaves? | Fixed-size sliding window |
| Do I need a shortest consecutive interval reaching a positive total, with non-negative values? | Expand/shrink variable window |
| Does the problem have contiguous items but no safe movement rule? | Do not force a sliding window; investigate another method |

### Common errors
- **Sorting daily counts:** destroys which days were consecutive.
- **Using `left <= right` for a target pair:** can reuse one position.
- **Starting the best sum at 0:** fails for all-negative windows.
- **Using one shrink instead of repeated shrinking:** can miss shorter windows.
- **Calling a sort-plus-scan O(n):** ignores sorting cost.
- **Using the positive-total shrink rule with negative values:** can miss answers.


## 15. Team practice: solve before looking at the answer checks

Work in pairs. One person traces; the other checks the movement rule. Swap roles.

### A — Sorted pair
List: `[1, 3, 4, 6, 8]`. Target: `10`.
1. Give the first sum and pointer movement.
2. Trace to one valid pair of indexes.
3. Explain why the discarded endpoint could not help.

### B — Fixed window: library loans
Daily counts: `[4, 2, 7, 1, 5, 3]`. Find the largest total over exactly three consecutive days.
1. Write all expected window totals by hand.
2. Show each remove/add update.
3. Why would sorting the counts invalidate the task?

### C — Variable window
Counts: `[1, 2, 3, 4]`. Target: `6`.
1. Find the shortest qualifying length by hand.
2. Show where the left pointer must move more than once.
3. Explain the non-negative assumption.

### D — Reject the wrong method
You need the shortest interval reaching a target, but values can be negative. Can you safely use our variable-window function? Explain using the counterexample in Section 13.

For one exercise, write: **Restate → Examples → Plan → Test → Reflect**. Do not submit only the final answer.

In [12]:
# Team workspace. Write expected results BEFORE calling the functions.
# Replace the comments with your trace, assertions, and reflection.

# A: expected pair indexes = ...
# B: expected window totals = ...; expected maximum = ...
# C: expected shortest length = ...
# D: why the method is not valid = ...

# Add your own edge-case test for each valid method.

## 16. Answer checks and presentation questions
### Read only after attempting
- **A:** `1 + 8 = 9`, too small → move left. `3 + 8 = 11`, too large → move right. `3 + 6 = 9`, too small → move left. `4 + 6 = 10` → indexes `(2, 3)`.
- **B:** Totals are `13, 10, 13, 9`; maximum is **13**. Sorting changes the day order and therefore changes the intervals.
- **C:** Answer **2**, from `[3, 4]`. At right index 3 the total is 9; removing 2 leaves 7, so shrink again after recording `[3, 4]`.
- **D:** No. With `[1, -1, 5]` and target 5, the method misses the length-1 interval `[5]`.

### Ask the class
1. What does a pointer store?
2. How is a target pair different from a contiguous window?
3. Why must the pair input already be sorted?
4. Why is `right - left + 1` the inclusive window length?
5. Why can a nested loop here still be O(n)?
6. Which method supports negative values: fixed-size sums or our variable target window?
7. What extra cost appears if you sort first?
8. Give a test that catches setting the best window sum to zero.

**Presenter answer cues:** Index; pairs may skip positions/windows cannot; safe endpoint elimination; count both ends; each pointer moves at most n times; fixed-size sums; O(n log n) sort and changed indexes; `[-4, -2, -7]`, k=2 → -6.

## 17. Exit ticket: show that you can choose, not just copy

Without running code, choose the method and justify it in one sentence:

| Task | Your method and reason |
|---|---|
| Find two prices adding to 50 in a sorted list. | … |
| Find the busiest five consecutive days. | … |
| Find the fewest consecutive days with at least 100 loans; counts are non-negative. | … |
| Find the shortest target-sum interval when values may be negative. | … |

Then explain one solution to a teammate using this checklist:
- [ ] I identified sorted order or contiguity, where required.
- [ ] I stated the assumptions needed for safe pointer movement.
- [ ] I traced an example without restarting the scan.
- [ ] I tested normal, empty/invalid, and boundary cases as appropriate.
- [ ] I explained time cost by counting total movements.

**Answer check:** Opposite-end two pointers; fixed-size window; variable-size window under the stated rules; not our simple variable-window method.

### Takeaway
**Move a boundary only when you can explain why it is safe. Keep useful information instead of recalculating it.**